# Ungraded Lab: Hyperparameter Tuning for Random Forest on Titanic

In this lab you will practice hyperparameter tuning for a **RandomForestClassifier** on the **Titanic** dataset. You will first train a baseline model with pre-selected hyperparameters, then use **GridSearchCV** (analogous to Keras Tuner) to find the optimal set.

## Download and Prepare the Dataset

In [ ]:
import seaborn as sns, pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def load_titanic():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    return df

df = load_titanic()
X = df.drop('survived', axis=1)
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Baseline Performance

First train a baseline model with arbitrarily selected hyperparameters for comparison.

In [ ]:
# Build the baseline model with hardcoded hyperparameters
b_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)
b_model.fit(X_train_s, y_train)
b_model

As shown, we hardcoded all hyperparameters. You will see how to automatically tune these next.

Let's set up the evaluation metrics. Accuracy is used as the primary metric.

In [ ]:
NUM_EPOCHS = 5  # equivalent: number of CV folds

b_scores = cross_val_score(b_model, X_train_s, y_train, cv=NUM_EPOCHS)
print(f"Baseline CV Accuracy: {b_scores.mean():.4f} (+/- {b_scores.std():.4f})")

In [ ]:
b_eval = {'accuracy': accuracy_score(y_test, b_model.predict(X_test_s))}
print(f"Baseline Test Accuracy: {b_eval['accuracy']:.4f}")

In [ ]:
def print_results(model, model_name, eval_dict):
    print(f'\n{model_name}:')
    print(f"n_estimators: {model.n_estimators}")
    print(f"max_depth: {model.max_depth}")
    print(f"min_samples_split: {model.min_samples_split}")
    for key, value in eval_dict.items():
        print(f'{key}: {value:.4f}')

print_results(b_model, 'BASELINE MODEL', b_eval)

## GridSearchCV — Automated Hyperparameter Tuning

To perform hypertuning with **GridSearchCV** you need to:
- Define the model
- Select which hyperparameters to tune
- Define the search space
- Define the search strategy

### Install and Import Packages

In [ ]:
# scikit-learn's GridSearchCV is already available
from sklearn.model_selection import GridSearchCV

### Define the Hypermodel

The search space below tunes:
- `n_estimators`: number of trees (50–200)
- `max_depth`: tree depth (None, 5, 10)
- `min_samples_split`: minimum samples to split a node
- `min_samples_leaf`: minimum samples in leaf nodes
- `max_features`: number of features per split

In [ ]:
def model_builder(params):
    '''Builds and returns a RandomForestClassifier with given hyperparameters.'''
    return RandomForestClassifier(
        n_estimators=params.get('n_estimators', 100),
        max_depth=params.get('max_depth', None),
        min_samples_split=params.get('min_samples_split', 2),
        min_samples_leaf=params.get('min_samples_leaf', 1),
        random_state=42
    )

# Define the hyperparameter search space
param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [None, 5, 10],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

## Instantiate the Tuner and Perform Hypertuning

**GridSearchCV** is analogous to Keras Tuner's Hyperband. It systematically searches all combinations of hyperparameters using cross-validation.

In [ ]:
# Instantiate GridSearchCV (analogous to kt.Hyperband)
tuner = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,                    # 5-fold cross validation
    scoring='accuracy',      # objective: maximize accuracy
    n_jobs=-1,               # use all CPU cores
    verbose=1
)
print("Search space summary:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
stop_early = 0.001  # minimum improvement threshold (analogous to EarlyStopping patience)

Now run the hyperparameter search. This will take a few minutes.

In [ ]:
# Perform hypertuning
tuner.fit(X_train_s, y_train)

In [ ]:
# Get the optimal hyperparameters
best_hps = tuner.best_params_

print(f"""
The hyperparameter search is complete.
Optimal n_estimators:      {best_hps['n_estimators']}
Optimal max_depth:         {best_hps['max_depth']}
Optimal min_samples_split: {best_hps['min_samples_split']}
Optimal min_samples_leaf:  {best_hps['min_samples_leaf']}
Best CV accuracy:          {tuner.best_score_:.4f}
""")

## Build and Train the Model

Rebuild the hypermodel with the best hyperparameters and retrain.

In [ ]:
# Build the model with the optimal hyperparameters
h_model = tuner.best_estimator_
h_model.summary = lambda: print(f"RandomForest(n_estimators={h_model.n_estimators}, max_depth={h_model.max_depth})")
print(f"Best model: {h_model}")

In [ ]:
# Cross-validate the hypertuned model
h_scores = cross_val_score(h_model, X_train_s, y_train, cv=NUM_EPOCHS)
print(f"Hypertuned CV Accuracy: {h_scores.mean():.4f} (+/- {h_scores.std():.4f})")

In [ ]:
# Evaluate the hypertuned model on the test set
h_eval = {'accuracy': accuracy_score(y_test, h_model.predict(X_test_s))}
print(f"Hypertuned Test Accuracy: {h_eval['accuracy']:.4f}")

Compare the results with the baseline model.

In [ ]:
print_results(b_model, 'BASELINE MODEL', b_eval)
print_results(h_model, 'HYPERTUNED MODEL', h_eval)

In [ ]:
# Plot CV score distribution
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(['Baseline','Hypertuned'], [b_scores.mean(), h_scores.mean()],
       yerr=[b_scores.std(), h_scores.std()], capsize=5, color=['steelblue','coral'])
ax.set_ylabel('CV Accuracy')
ax.set_title('Baseline vs Hypertuned Random Forest — Titanic')
plt.tight_layout()
plt.show()